In [11]:
import pandas as pd

raw = pd.read_csv("Exam_Score_Prediction.csv")

In [12]:
raw_trunc = raw.drop(columns=["student_id", "age", "gender", "course", "internet_access", "sleep_hours", "exam_difficulty"])

raw_trunc.head()

,study_hours,class_attendance,sleep_quality,study_method,facility_rating,exam_score
0,2.78,92.9,poor,coaching,low,58.9
1,3.37,64.8,average,online videos,medium,54.8
2,7.88,76.8,poor,coaching,high,90.3
3,0.67,48.4,average,online videos,low,29.7
4,0.89,71.6,poor,coaching,low,43.7


In [13]:
y = raw["exam_score"]

X = raw.drop(columns=["exam_score"])

In [18]:
numeric_columns = with_ohe.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = with_ohe.select_dtypes(exclude=["number"]).columns.tolist()



In [15]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [16]:
class IQRClipper(BaseEstimator, TransformerMixin):
    def __init__(self, k=1.5):
        self.k = k

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        q1 = np.nanpercentile(X, 25, axis=0)
        q3 = np.nanpercentile(X, 75, axis=0)
        iqr = q3 - q1
        self.lower = q1 - self.k * iqr
        self.upper = q3 + self.k * iqr
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.clip(X, self.lower, self.upper)

In [19]:
num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clipper", IQRClipper(k=1.5)),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, numeric_columns),
        ("cat", cat_pipeline, categorical_columns)
    ]
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("regressor", Ridge(alpha=1.0))
])